# Дообучение YOLO11-OBB на MAR20 (военные самолёты, 20 типов)

**Настройки ноутбука Kaggle:** Accelerator — GPU T4 x2 или P100, Internet — On (нужен подтверждённый телефон).

**Данные:** Add Input → полный MAR20 (3842 снимка, папки `Annotations/Oriented Bounding Boxes`, `ImageSets/Main`, `JPEGImages`).
Путь найдётся автоматически. Официальный архив: https://gcheng-nwpu.github.io/ (CC BY-NC 4.0) —
при необходимости загрузить его как свой Kaggle Dataset. Зеркало на Hugging Face содержит только 100 снимков — не подходит.

**Запуск в фоне:** Save Version → Save & Run All. Веса, метрики и графики окажутся в Output версии (`/kaggle/working/export`).

Старт с весов `yolo11s-obb.pt` (Ultralytics, DOTA-v1.0, AGPL-3.0) — трансферное обучение.

In [ ]:
REPO_URL = "https://github.com/miss-mississippi/muxxed_defence_tech.git"
MAR20_SRC = None          # None — найти MAR20 в /kaggle/input автоматически
MODEL = "yolo11s-obb.pt"  # yolo11m-obb.pt — точнее, но ~2x медленнее
EPOCHS, IMGSZ, BATCH = 60, 800, 16  # снимки MAR20 800x800
RUN_NAME = "mar20_s_800"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/repo && git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q ultralytics==8.4.152

In [ ]:
import json
from pathlib import Path

if MAR20_SRC is None:
    hits = sorted(Path("/kaggle/input").glob("**/Annotations/Oriented Bounding Boxes"))
    if not hits:
        raise SystemExit("MAR20 не найден в /kaggle/input: Add Input → полный датасет MAR20")
    MAR20_SRC = str(hits[0].parent.parent)
print("MAR20:", MAR20_SRC)

!python scripts/prepare_mar20.py --src "$MAR20_SRC" --out /kaggle/working/mar20_yolo

stats = json.loads(Path("/kaggle/working/mar20_yolo/stats.json").read_text())
total = sum(stats["images"].values())
assert total > 3000, f"в датасете {total} снимков — ожидалось 3842, проверьте источник"

In [ ]:
import torch
DEVICE = ",".join(str(i) for i in range(torch.cuda.device_count())) or "cpu"
print("device:", DEVICE)
!python scripts/train.py --data /kaggle/working/mar20_yolo/mar20.yaml --weights $MODEL --name $RUN_NAME \
    --epochs $EPOCHS --imgsz $IMGSZ --batch $BATCH --device $DEVICE --workers 4

In [ ]:
import glob, shutil
from IPython.display import Image, Markdown, display

export = Path("/kaggle/working/export")
export.mkdir(exist_ok=True)
run_dir = Path("runs/obb") / RUN_NAME
for f in [f"weights/{RUN_NAME}.pt", *glob.glob(f"outputs/eval/{RUN_NAME}_*.*"), run_dir / "results.csv", *run_dir.glob("*.png")]:
    shutil.copy2(f, export)
for md in glob.glob(f"outputs/eval/{RUN_NAME}_*.md"):
    display(Markdown(Path(md).read_text()))
display(Image(str(run_dir / "results.png")))
print(sorted(p.name for p in export.iterdir()))